# 03 — Persiapan Prediksi dan Submission

Notebook ini menyiapkan pipeline inference untuk submission Codabench.

**Status saat ini:** Fase training. Data evaluasi resmi (`val_...` dan `test_...`) **belum tersedia**.

Notebook ini akan:
1. Memuat model terbaik dari checkpoint
2. Menyiapkan pipeline prediksi yang siap digunakan
3. Memvalidasi output CSV sebelum submission
4. Membuat ZIP untuk submission

**⚠ Jangan jalankan pembuatan submission sekarang jika data target belum tersedia.**

## 1. Import dan Konfigurasi

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

import torch

# Tambahkan folder src
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

from data_utils import (
    MODELS_DIR, PREDICTIONS_DIR, DATA_DIR,
    NUM_CLASSES, validate_submission_csv,
    get_label_to_character_map, load_classes
)
from train import ImprovedCNN, BaselineCNN, get_device, load_model
from predict import predict_folder, save_predictions, create_submission_zip

print(f"Project root: {PROJECT_ROOT}")

## 2. Muat Konfigurasi Inference

In [ ]:
# Muat konfigurasi yang disimpan saat training
config_path = os.path.join(MODELS_DIR, 'inference_config.json')

if not os.path.exists(config_path):
    print(" File inference_config.json belum ada.")
    print("  Jalankan notebook 02_training.ipynb terlebih dahulu.")
    raise FileNotFoundError(f"Konfigurasi inference tidak ditemukan: {config_path}")

with open(config_path, 'r', encoding='utf-8') as f:
    config = json.load(f)

print("Konfigurasi inference:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 3. Muat Model Terbaik

In [ ]:
# Deteksi device
device = get_device()

# Tentukan kelas model berdasarkan konfigurasi
model_class_name = config['model_class']
if model_class_name == 'ImprovedCNN':
    model_class = ImprovedCNN
    model_kwargs = {'num_classes': config['num_classes'], 'dropout_rate': 0.4}
elif model_class_name == 'BaselineCNN':
    model_class = BaselineCNN
    model_kwargs = {'num_classes': config['num_classes']}
else:
    raise ValueError(f"Model class tidak dikenal: {model_class_name}")

# Muat checkpoint
checkpoint_path = os.path.join(MODELS_DIR, config['model_checkpoint'])
model = load_model(model_class, checkpoint_path, device, **model_kwargs)
print("\n✓ Model berhasil dimuat dan siap untuk prediksi.")

## 4. Pipeline Prediksi

### Cara penggunaan:

1. **Fase 1:** Ganti `TARGET_DIR` dengan path ke folder yang berisi gambar `val_*.png`
2. **Fase 2:** Ganti `TARGET_DIR` dengan path ke folder yang berisi gambar `test_*.png`

Pipeline akan:
- Membaca semua gambar PNG di folder target
- Mengambil ID dari nama file (tanpa `.png`)
- Melakukan preprocessing yang sama dengan training
- Menghasilkan prediksi label 0-14
- Menyimpan `predictions.csv` dengan header `id,label`

In [ ]:
# ====================================================================
# KONFIGURASI: Ubah path ini sesuai fase kompetisi
# ====================================================================

# Fase 1: folder berisi gambar val_*.png
# TARGET_DIR = os.path.join(DATA_DIR, "val")

# Fase 2: folder berisi gambar test_*.png 
# TARGET_DIR = os.path.join(DATA_DIR, "test")

# Untuk saat ini, kita belum punya data evaluasi resmi
TARGET_DIR = None  # Ubah ini ketika data tersedia

# ====================================================================

In [ ]:
# Cek apakah folder target tersedia
if TARGET_DIR is None:
    print("="*60)
    print("ℹ TARGET_DIR belum diset.")
    print()
    print("Data evaluasi resmi belum tersedia.")
    print("Ketika data dirilis, ubah variabel TARGET_DIR di sel sebelumnya")
    print("ke path folder yang berisi gambar evaluasi, lalu jalankan sel-sel")
    print("berikutnya.")
    print()
    print("Contoh:")
    print("  Fase 1: TARGET_DIR = os.path.join(DATA_DIR, 'val')")
    print("  Fase 2: TARGET_DIR = os.path.join(DATA_DIR, 'test')")
    print("="*60)
elif not os.path.isdir(TARGET_DIR):
    print(f"⚠ Folder target tidak ditemukan: {TARGET_DIR}")
    print("Pastikan path sudah benar dan folder berisi gambar PNG.")
else:
    # Hitung jumlah gambar
    png_count = len([f for f in os.listdir(TARGET_DIR) if f.endswith('.png')])
    print(f"✓ Folder target ditemukan: {TARGET_DIR}")
    print(f"  Jumlah gambar PNG: {png_count}")
    
    if png_count == 0:
        print("⚠ Tidak ada gambar PNG di folder target.")

In [ ]:
# Jalankan prediksi (hanya jika folder target tersedia dan berisi gambar)
if TARGET_DIR is not None and os.path.isdir(TARGET_DIR):
    png_files = [f for f in os.listdir(TARGET_DIR) if f.endswith('.png')]
    
    if len(png_files) > 0:
        print("Memulai prediksi...")
        results_df = predict_folder(model, TARGET_DIR, device, batch_size=64)
        
        if results_df is not None:
            # Tampilkan preview
            print("\nPreview prediksi:")
            print(results_df.head(10))
            print(f"\nDistribusi label prediksi:")
            print(results_df['label'].value_counts().sort_index())
    else:
        print("⚠ Folder target kosong, tidak ada gambar untuk diprediksi.")
        results_df = None
else:
    print("ℹ Prediksi dilewati karena folder target belum tersedia.")
    results_df = None

## 5. Simpan dan Validasi Predictions CSV

In [ ]:
if results_df is not None:
    # Simpan predictions.csv
    csv_path = save_predictions(results_df)
    
    # Validasi
    is_valid, issues = validate_submission_csv(csv_path)
    
    if is_valid:
        print("\n✓ CSV valid untuk submission!")
        print(f"  Jumlah baris: {len(results_df)}")
        print(f"  Kolom: {list(results_df.columns)}")
        print(f"  Rentang label: [{results_df['label'].min()}, {results_df['label'].max()}]")
    else:
        print("\n✗ CSV TIDAK valid:")
        for issue in issues:
            print(f"  - {issue}")
else:
    print("ℹ Tidak ada hasil prediksi untuk disimpan.")

## 6. Buat ZIP Submission

In [ ]:
if results_df is not None:
    csv_path = os.path.join(PREDICTIONS_DIR, 'predictions.csv')
    
    if os.path.exists(csv_path):
        # Buat ZIP
        zip_path = create_submission_zip(csv_path)
        
        if zip_path:
            print(f"\n✓ File submission siap: {zip_path}")
            print("\nLangkah selanjutnya:")
            print("1. Upload file submission.zip ke Codabench")
            print("2. Pastikan format sudah sesuai (predictions.csv di root ZIP)")
        else:
            print("✗ Gagal membuat ZIP. Perbaiki masalah CSV terlebih dahulu.")
    else:
        print(f"⚠ File CSV belum dibuat: {csv_path}")
else:
    print("ℹ Tidak ada prediksi untuk di-ZIP.")
    print("  Setelah data evaluasi resmi tersedia:")
    print("  1. Set TARGET_DIR ke folder data evaluasi")
    print("  2. Jalankan sel prediksi di atas")
    print("  3. Jalankan sel ini untuk membuat ZIP submission")

---

## Catatan Penting

1. **Preprocessing harus konsisten** dengan saat training:
   - Gambar dibaca sebagai grayscale
   - Dinormalisasi ke [0, 1] dengan membagi 255
   - Shape input model: (batch, 1, 64, 64)

2. **Format submission Codabench:**
   - File ZIP berisi `predictions.csv` langsung di root (bukan di dalam folder)
   - CSV memiliki header persis `id,label`
   - Label berupa integer 0–14
   - Setiap ID target tercantum tepat satu kali

3. **Jangan membuat submission palsu** jika data evaluasi belum tersedia.